In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
df4 = pd.read_pickle('../data/df3_with_regions.pkl')
print(f"Loaded df3_with_regions: {df4.shape}")

Loaded df3_with_regions: (7235973, 95)


for visualisation suite) 
- function to call values of current transfer timing distribution by age group and time of day
- function to call distribution of spatial transfer location pairs
- temporal transfer pattern distribution (y axis average transfer time, x axis hour of day) 


In [3]:
# Cell 3: Verify filter value
print(df4['is_same_journey_final'].value_counts().head(10))

is_same_journey_final
False    5335081
True     1900892
Name: count, dtype: int64


In [4]:
# drop bad gap and mixed patron cards upfront
crds_bad_gap_vis = df4[
    (df4['is_last_stage'] == False) &
    (df4['time_gap_mins'].isna() | (df4['time_gap_mins'] < 0))
]['CRD_NUM'].unique()

df_positive_vis = df4[
    (df4['is_last_stage'] == False) &
    (df4['time_gap_mins'] > 0)
]
patron_per_card_vis = df_positive_vis.groupby('CRD_NUM')['PATRON_CATG_DESC_TXT'].nunique()
crds_mixed_patron_vis = patron_per_card_vis[patron_per_card_vis > 1].index.tolist()

crds_to_drop_vis = set(crds_bad_gap_vis) | set(crds_mixed_patron_vis)

# filter to confirmed transfers only (reusable base)
df_transfers = df4[
    (df4['is_same_journey_final'] == True) &
    (~df4['CRD_NUM'].isin(crds_to_drop_vis))
].copy()

df_transfers['age_group'] = df_transfers['PATRON_CATG_DESC_TXT'].map({
    'Student':        '7-19',
    'Adult':          '20-59',
    'Senior Citizen': '60+'
})

df_transfers['hour_of_day'] = df_transfers['next_ENTRY_TM'].dt.hour

# filter out train-train and other mode pairs
df_transfers = df_transfers[
    df_transfers['mode_pair'].isin(['bus_bus', 'bus_train', 'train_bus'])
].copy()

print(f"Confirmed transfers (excl. train-train/other): {len(df_transfers):,}")
print(f"Null age_group: {df_transfers['age_group'].isna().sum():,}")
print(f"Null hour_of_day: {df_transfers['hour_of_day'].isna().sum():,}")
print(f"Null time_gap_mins: {df_transfers['time_gap_mins'].isna().sum():,}")
print(df_transfers['mode_pair'].value_counts())

Confirmed transfers (excl. train-train/other): 1,809,440
Null age_group: 0
Null hour_of_day: 0
Null time_gap_mins: 0
mode_pair
bus_train    671137
train_bus    615833
bus_bus      522470
Name: count, dtype: int64


In [5]:
df_transfers.head()

,BUS_SVC_NUM,CRD_NUM,DEST_LOC_ID_NUM,ENTRY_DT,ENTRY_TM,EXIT_DT,EXIT_TM,JRNY_ID_NUM,ORIG_LOC_ID_NUM,RIDE_DISC_AMT,...,(BTS_Only)_round_trip_reason,BTS_flag_nodist,BTS_flag_reason_nodist,is_same_journey_final,is_same_journey_final_nodist,orig_region,dest_region,house_area,age_group,hour_of_day
0,45.0,100006223599,3968.0,2025-02-12,2025-02-12 06:49:58,2025-02-12,2025-02-12 07:00:33,110715426253,4009.0,0.0,...,different_origin_destination,False,candidate_transfer,True,True,HOUGANG,SERANGOON,HOUGANG,7-19,7
2,NaN,100006223599,106.0,2025-02-12,2025-02-12 15:21:06,2025-02-12,2025-02-12 15:32:56,110713665019,109.0,0.0,...,different_origin_destination,False,candidate_transfer,True,True,KALLANG,SERANGOON,HOUGANG,7-19,15
8,871.0,130013244638,2782.0,2025-02-12,2025-02-12 06:39:37,2025-02-12,2025-02-12 06:53:26,110713970377,6011.0,0.0,...,different_origin_destination,False,candidate_transfer,True,True,TENGAH,BUKIT BATOK,TENGAH,20-59,6
9,NaN,130013244638,45.0,2025-02-12,2025-02-12 06:57:56,2025-02-12,2025-02-12 07:19:52,110713970377,29.0,0.5,...,different_origin_destination,False,candidate_transfer,True,True,BUKIT BATOK,WOODLANDS,TENGAH,20-59,7
11,901.0,130013244638,5012.0,2025-02-12,2025-02-12 15:14:46,2025-02-12,2025-02-12 15:21:56,110714090504,3173.0,0.0,...,different_origin_destination,False,candidate_transfer,True,True,WOODLANDS,WOODLANDS,TENGAH,20-59,15


In [6]:
# Cell 5: Function 1 — Transfer timing distribution by age group and hour of day
# Output: avg transfer time (time_gap_mins) per age group × hour
# CSV columns: age_group, hour_of_day, avg_transfer_time_mins, count
# I included count, in each aggregation s.t. you can recalculate averages if you want to aggregate on something else

def trf_time_distribution_csv(df_trf):
    result = (
        df_trf.groupby(['age_group', 'hour_of_day'])['time_gap_mins']
        .agg(avg_transfer_time_mins='mean', count='size')
        .reset_index()
        .sort_values(['age_group', 'hour_of_day'])
        .reset_index(drop=True)
    )
    result.to_csv('../data/trf_time_distribution.csv', index=False)
    print(f"Saved trf_time_distribution.csv: {result.shape}")
    return result

def trf_time_distribution(file='../data/trf_time_distribution.csv'):
    return pd.read_csv(file)

In [7]:
# Cell 6: Run Function 1
trf_time_dist_df = trf_time_distribution_csv(df_transfers)
print(trf_time_dist_df)

Saved trf_time_distribution.csv: (57, 4)
   age_group  hour_of_day  avg_transfer_time_mins   count
0      20-59            5                4.001962    7417
1      20-59            6                3.755705   63674
2      20-59            7                3.894497  151973
3      20-59            8                3.759408  171146
4      20-59            9                4.378331   86280
5      20-59           10                4.752689   42326
6      20-59           11                5.043031   35129
7      20-59           12                5.331647   35486
8      20-59           13                5.656247   36218
9      20-59           14                5.642813   34700
10     20-59           15                5.563386   38601
11     20-59           16                5.350605   49996
12     20-59           17                4.258018   84958
13     20-59           18                4.574760  137011
14     20-59           19                4.908569   94093
15     20-59           20      

In [8]:
# Cell 7: Function 2 — Transfer volume by origin-destination region pair and hour of day
# Output: count of transfers per orig_region × dest_region × hour_of_day
# CSV columns: orig_region, dest_region, hour_of_day, count

def trf_region_pair_csv(df_trf):
    result = (
        df_trf.groupby(['orig_region', 'dest_region', 'hour_of_day'])
        .size()
        .reset_index(name='count')
        .sort_values(['orig_region', 'dest_region', 'hour_of_day'])
        .reset_index(drop=True)
    )
    result.to_csv('../data/trf_region_pair.csv', index=False)
    print(f"Saved trf_region_pair.csv: {result.shape}")
    return result

def trf_region_pair(file='../data/trf_region_pair.csv'):
    return pd.read_csv(file)

In [9]:
# Cell 8: Run Function 2
trf_region_pair_df = trf_region_pair_csv(df_transfers)
print(trf_region_pair_df.head(10))

Saved trf_region_pair.csv: (26253, 4)
  orig_region dest_region  hour_of_day  count
0  ANG MO KIO  ANG MO KIO            5    392
1  ANG MO KIO  ANG MO KIO            6   2036
2  ANG MO KIO  ANG MO KIO            7   3822
3  ANG MO KIO  ANG MO KIO            8   3900
4  ANG MO KIO  ANG MO KIO            9   2299
5  ANG MO KIO  ANG MO KIO           10   1576
6  ANG MO KIO  ANG MO KIO           11   1424
7  ANG MO KIO  ANG MO KIO           12   1424
8  ANG MO KIO  ANG MO KIO           13   1400
9  ANG MO KIO  ANG MO KIO           14   1573


In [10]:
# Frontend query functions (read from pre-computed CSVs)

_data_dir = '../data'

def get_trf_time_distribution(age_group=None, hour=None):
    """
    Returns avg transfer time distribution by age group and hour of day.

    Inputs:
    - age_group: None (all), or one of '7-19', '20-59', '60+'
    - hour:      None (all), or int 0-23

    Returns: DataFrame with [age_group, hour_of_day, avg_transfer_time_mins]
    """
    df = pd.read_csv(os.path.join(_data_dir, 'trf_time_distribution.csv'))

    if age_group is not None:
        df = df[df['age_group'] == age_group]
    if hour is not None:
        df = df[df['hour_of_day'] == hour]

    return df.reset_index(drop=True)


def get_trf_region_pair(orig_region=None, dest_region=None, hour=None):
    """
    Returns transfer volume by origin-destination region pair and hour of day.

    Inputs:
    - orig_region: None (all), or filter by origin region string
    - dest_region: None (all), or filter by destination region string
    - hour:        None (all), or integer 0-23

    Returns: DataFrame with [orig_region, dest_region, hour_of_day, count]
    """
    df = pd.read_csv(os.path.join(_data_dir, 'trf_region_pair.csv'))

    if orig_region is not None:
        df = df[df['orig_region'] == orig_region]
    if dest_region is not None:
        df = df[df['dest_region'] == dest_region]
    if hour is not None:
        df = df[df['hour_of_day'] == hour]

    return df.reset_index(drop=True)

def get_trf_temporal_pattern(patron=None):
    """
    Returns avg transfer time by hour of day for the temporal pattern chart.

    Inputs:
    - patron: None (all), or one of 'Adult', 'Student', 'Senior Citizen'

    Returns: DataFrame with [hour_of_day, PATRON_CATG_DESC_TXT, avg_transfer_time_mins, count]
    """
    age_group_map = {
        '7-19':  'Student',
        '20-59': 'Adult',
        '60+':   'Senior Citizen'
    }

    df = pd.read_csv(os.path.join(_data_dir, 'trf_time_distribution.csv'))
    df['PATRON_CATG_DESC_TXT'] = df['age_group'].map(age_group_map)

    if patron is not None:
        df = df[df['PATRON_CATG_DESC_TXT'] == patron]

    return df[['hour_of_day', 'PATRON_CATG_DESC_TXT', 'avg_transfer_time_mins', 'count']].reset_index(drop=True)